In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import pickle
import os
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
import warnings
from umap import UMAP
import yaml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from utils.mypath import MyPath
from utils.mypalette import mutation_palette, condition_palette, day_palette, cell_line_palette
from utils.snn import *
from utils.start_notebook import fix_embedding_format
from sklearnex import patch_sklearn
patch_sklearn()
  
warnings.filterwarnings('ignore')


#single_exp = ['E1', 'E2', 'E3', 'E4']
simclr_exp = ['240517_11', '240813_5']
exp_names = {'240517_11': 'Standard\nSimCLR',
             '240813_5': 'Augmented\nSimCLR',}
res_dir = '../results'
config_dir = '../configs'

order = ['Hand-crafted', 'Pretrained ResNet18', 'Standard SimCLR', 'uniDINO', 'CellPaint SSL', 'Augmented SimCLR']
days = ['E1', 'E2', 'E3', 'E4']

img_source_dir = '/path/to/image/folder'
global_annotations = pd.read_csv('../data/crop16_EF_2405.csv')
global_annotations['file_name'] = global_annotations['img_code'].apply(lambda x: os.path.basename(x).split('.')[0])

pca = PCA(n_components=20, random_state=0)

In [ ]:
UMAP_features = {}
PCA_features = {}
PCA_var_explained = {}

for e in simclr_exp:
    if e.__contains__('24'):
        config = yaml.safe_load(open(os.path.join(config_dir, e, 'simclr_acs.yml')))
        temp_annotations = pd.read_csv(os.path.join(MyPath.db_root_dir(config['train_db_name'])))

        if len(temp_annotations) > len(global_annotations):
            temp_annotations = temp_annotations[temp_annotations['img_code'].isin(global_annotations['img_code'])]
        
        assert temp_annotations['img_code'].equals(global_annotations['img_code'])

        with open(os.path.join(res_dir, e, 'latent_embeddings_train.pickle'), 'rb') as f:
            raw_emb = pickle.load(f)

        if config['pairing_strategy'] == 'mut_cond':
            temp_annotations = temp_annotations.reset_index(drop=True)

        raw_emb = raw_emb[temp_annotations.index]
        #UMAP_features[e] = raw_emb
        name = exp_names[e]
        scaled_emb = StandardScaler().fit_transform(raw_emb)
        UMAP_features[name] = UMAP(n_components=2, random_state=0, n_neighbors=15).fit_transform(scaled_emb)
        PCA_features[name] = pca.fit_transform(scaled_emb)

handcrafted_df = pd.read_csv('../embeddings/classical-features.csv')
# Extract file names (without extension) from img_code columns

assert handcrafted_df['file_name'].equals(global_annotations['file_name'])

handcrafted_feat = handcrafted_df.drop(['img_code', 'mutation', 'condition', 'experiment', 'file_name', 'als_label'], axis=1)
handcrafted_feat = handcrafted_feat.loc[:, handcrafted_feat.var() != 0].values
handcrafted_feat = StandardScaler().fit_transform(handcrafted_feat)
UMAP_features['Hand-crafted'] = UMAP(n_components=2, random_state=0, n_neighbors=15).fit_transform(handcrafted_feat)

pretrain_resnet_annotations = pd.read_csv(os.path.join('../embeddings', 'pretrained_resnet_embeddings.csv'))
assert pretrain_resnet_annotations['file_name'].equals(global_annotations['file_name'])

pretrain_resnet_annotations['parsed_embedding'] = pretrain_resnet_annotations['pretrained_resnet_embedding'].apply(fix_embedding_format)
pretrained_emb = np.stack(pretrain_resnet_annotations['parsed_embedding'].values)
#UMAP_features['pretrained resnet18'] = pretrained_emb
scaled_pretrained = StandardScaler().fit_transform(pretrained_emb)
UMAP_features['Pretrained\nResNet18'] = UMAP(n_components=2, random_state=0, n_neighbors=15).fit_transform(scaled_pretrained)

# Prepare uniDINO embeddings. Two versions: one with all dimensions and one without low-variance dimensions
low_variance_threshold = 1e-5
uni_emb = pd.read_csv(os.path.join('../embeddings', 'uniDINO_crop_nopostproc.csv'))
assert uni_emb['file_name'].equals(global_annotations['file_name'])

uni_emb_feat = uni_emb.drop([c for c in uni_emb.columns if 'emb' not in c], axis=1)
variances = uni_emb_feat.var(axis=0)
high_variance_mask = variances > low_variance_threshold
scaled_uni_highvar = StandardScaler().fit_transform(uni_emb_feat.values[:, high_variance_mask])
UMAP_features['uniDINO'] = UMAP(n_components=2, random_state=0, n_neighbors=15).fit_transform(scaled_uni_highvar)


cellpaint_df = pd.read_csv(os.path.join('../embeddings', 'CellPaint_SSL.csv'))
cellpaint_emb = cellpaint_df.drop([c for c in cellpaint_df.columns if 'emb' not in c], axis=1)
assert cellpaint_df['file_name'].equals(global_annotations['file_name'])
variances = cellpaint_emb.var(axis=0)
high_variance_mask = variances > low_variance_threshold
scaled_cellpaint_highvar = StandardScaler().fit_transform(cellpaint_emb.values[:, high_variance_mask])
UMAP_features['CellPaint SSL'] = UMAP(n_components=2, random_state=0, n_neighbors=15).fit_transform(scaled_cellpaint_highvar)


# Define the labels for coloring
labels = ['cell_line', 'mutation', 'condition', 'experiment']
order = ['Hand-crafted', 'Pretrained\nResNet18', 'Standard\nSimCLR', 'uniDINO', 'CellPaint SSL', 'Augmented\nSimCLR']
UMAP_features = {name: UMAP_features[name] for name in order if name in UMAP_features}
embeddings_keys = list(UMAP_features.keys())
palettes={'cell_line': cell_line_palette, 'mutation': mutation_palette, 'condition': condition_palette, 'experiment': day_palette}

In [ ]:
# Create a figure with subplots
fig, axes = plt.subplots(len(embeddings_keys), len(labels), figsize=(20, 25), constrained_layout=True)
print(axes.shape)
plt.subplots_adjust(wspace=0.4, hspace=0.6)

for i, embedding_key in enumerate(order):
    for j, label in enumerate(labels):
        ax = axes[i, j]
        scatter = ax.scatter(
            UMAP_features[embedding_key][:, 0], 
            UMAP_features[embedding_key][:, 1], 
            c=global_annotations[label].map(
                palettes[label] if label in palettes else {k: k for k in global_annotations[label].unique()}
            ), 
            s=4, 
            alpha=0.4,
            rasterized=True,
        )
        if j == 0:
            ax.text(-0.1, 0.5, embedding_key, va='center', ha='right', rotation=90, transform=ax.transAxes, fontsize=28, fontweight='bold')
        ax.axis('off')
        
plt.savefig(os.path.join('../results', 'all_UMAP_zscore.pdf'), dpi=300, bbox_inches='tight')
plt.show()